# Figure generation (paper)

This notebook contains one cell per figure used in the paper. Running a single figure cell only saves that figure.
Outputs are written to `paper/figures/results/` as PDF+PNG.

Tip: run the Setup + Helpers cells once, then run each figure cell.


In [19]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter


In [20]:
# --- Setup (robust to notebook working directory) ---
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'results').exists():
    ROOT = ROOT.parent
if not (ROOT / 'results').exists():
    raise RuntimeError('Could not locate project root (expected to find results/). Please run the notebook from inside the repository.')

RESULTS_DIR = ROOT / 'results'
RESULTS_RELATED_DIR = ROOT / 'results_related'
OUT_DIR = ROOT / 'plot'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUNS = [
    {
        'name': 'FL-TFlow',
        'color': '#1f77b4',
        'path': RESULTS_DIR / 'FINAL_PAPER_LoRA16_FedProx_0005_Orig_R30_FullTest',
    },
    {
        'name': 'LogBERT',
        'color': '#d62728',
        'path': RESULTS_RELATED_DIR / 'RELATED_LogBERT_Edge_FedAvg_R30',
    },
]

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 18,
    'axes.labelsize': 20,
    'axes.titlesize': 20,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'lines.linewidth': 3.0,
    'figure.figsize': (9.0, 6.2),
})

In [21]:
# --- Helpers ---
def read_csv(run, rel):
    p = run['path'] / rel
    return pd.read_csv(p)

def save(fig, stem):
    fig.savefig(OUT_DIR / f'{stem}.pdf', bbox_inches='tight')
    fig.savefig(OUT_DIR / f'{stem}.png', dpi=300, bbox_inches='tight')

def format_percent_axis(ax, decimals=3):
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:.{decimals}f}'))
    ax.yaxis.offsetText.set_visible(False)


## Fig 1a - F1 over rounds (k=10)


In [22]:
k = 10
fig, ax = plt.subplots()

for run in RUNS:
    df = read_csv(run, 'eval_fpr_target_flow/f1_scores_fpr_target.csv')
    df = df[(df['granularity'] == 'flow') & (df['k'] == k)].sort_values('round')
    ax.plot(df['round'], df['f1_score'], color=run['color'], marker='o', label=run['name'])

    best = df.loc[df['f1_score'].idxmax()]
    ax.scatter(float(best['round']), float(best['f1_score']),
               color=run['color'], s=160, marker='*', zorder=10,
               edgecolors='black', linewidths=0.8)

ax.set_xlabel('Round')
ax.set_ylabel('F1 Score')
ax.set_ylim(-0.02, 1.02)
ax.grid(True, linestyle='--', alpha=0.6)

# Legend: move a bit up/down by changing bbox_to_anchor y
ax.legend(loc='lower right', bbox_to_anchor=(0.98, 0.08), frameon=True)

save(fig, 'fig1a_f1_over_rounds_k10')
plt.close(fig)


## Fig 1b - Benign FPR over rounds (k=10)


In [23]:
k = 10
fig, ax = plt.subplots()

for run in RUNS:
    df = read_csv(run, 'eval_fpr_target_flow/f1_scores_fpr_target.csv')
    df = df[(df['granularity'] == 'flow') & (df['k'] == k)].sort_values('round')
    ax.plot(df['round'], df['benign_fpr'] * 100.0,
            color=run['color'], marker='s', label=run['name'])

ax.set_xlabel('Round')
ax.set_ylabel('Benign FPR (%)')
ax.grid(True, linestyle='--', alpha=0.6)
format_percent_axis(ax, decimals=3)
ax.legend(loc='upper right', frameon=True)

save(fig, 'fig1b_benign_fpr_over_rounds_k10')
plt.close(fig)


## Fig 2a - F1 by k (Round 30)


In [24]:
round_num = 30
ks = [1, 3, 5, 10]
width = 0.36
xs = list(range(len(ks)))

fig, ax = plt.subplots()

for i, run in enumerate(RUNS):
    df = read_csv(run, 'eval_fpr_target_flow/f1_scores_fpr_target.csv')
    df = df[(df['granularity'] == 'flow') & (df['round'] == round_num)]
    df = df[df['k'].isin(ks)].sort_values('k')
    ys = [float(df[df['k'] == k].iloc[0]['f1_score']) for k in ks]

    offset = (i - 0.5) * width
    ax.bar([x + offset for x in xs], ys, width=width,
           color=run['color'], edgecolor='white', linewidth=0.6,
           label=run['name'])

ax.set_xticks(xs)
ax.set_xticklabels([str(k) for k in ks])
ax.set_xlabel('Top-k')
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.02)
ax.grid(True, axis='y', linestyle='--', alpha=0.6)

# Compact legend
ax.legend(loc='upper left', bbox_to_anchor=(0.01, 0.98),
          frameon=True, framealpha=0.88,
          borderpad=0.25, labelspacing=0.15,
          handletextpad=0.35, handlelength=1.2,
          fontsize=14)

save(fig, 'fig2a_f1_by_k_round30')
plt.close(fig)

## Fig 2b - Benign FPR by k (Round 30)


In [25]:
round_num = 30
ks = [1, 3, 5, 10]
width = 0.36
xs = list(range(len(ks)))

fig, ax = plt.subplots()

for i, run in enumerate(RUNS):
    df = read_csv(run, 'eval_fpr_target_flow/f1_scores_fpr_target.csv')
    df = df[(df['granularity'] == 'flow') & (df['round'] == round_num)]
    df = df[df['k'].isin(ks)].sort_values('k')
    ys = [float(df[df['k'] == k].iloc[0]['benign_fpr'] * 100.0) for k in ks]

    offset = (i - 0.5) * width
    ax.bar([x + offset for x in xs], ys, width=width,
           color=run['color'], edgecolor='white', linewidth=0.6,
           label=run['name'])

ax.set_xticks(xs)
ax.set_xticklabels([str(k) for k in ks])
ax.set_xlabel('Top-k')
ax.set_ylabel('Benign FPR (%)')
ax.grid(True, axis='y', linestyle='--', alpha=0.6)
format_percent_axis(ax, decimals=3)

ax.legend(loc='upper right', bbox_to_anchor=(0.99, 0.98),
          frameon=True, framealpha=0.88,
          borderpad=0.25, labelspacing=0.15,
          handletextpad=0.35, handlelength=1.2,
          fontsize=14)

save(fig, 'fig2b_benign_fpr_by_k_round30')
plt.close(fig)


## Fig 3a - Flow vs Window30s (F1, Round 30, k=10)


In [26]:
round_num = 30
k = 10
cats = ['Flow', 'Window 30s']
xs = list(range(len(cats)))
width = 0.36

fig, ax = plt.subplots()

for i, run in enumerate(RUNS):
    flow = read_csv(run, 'eval_fpr_target_flow/f1_scores_fpr_target.csv')
    flow = flow[(flow['granularity']=='flow') & (flow['round']==round_num) & (flow['k']==k)].iloc[0]
    win = read_csv(run, 'eval_fpr_target_window30s/f1_scores_fpr_target.csv')
    win = win[(win['granularity']=='window') & (win['round']==round_num) & (win['k']==k)].iloc[0]
    ys = [float(flow['f1_score']), float(win['f1_score'])]

    offset = (i - 0.5) * width
    ax.bar([x + offset for x in xs], ys, width=width,
           color=run['color'], edgecolor='white', linewidth=0.6,
           label=run['name'])

ax.set_xticks(xs)
ax.set_xticklabels(cats)
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.02)
ax.grid(True, axis='y', linestyle='--', alpha=0.6)
ax.legend(loc='upper right', bbox_to_anchor=(0.99, 0.98),
          frameon=True, framealpha=0.88,
          borderpad=0.25, labelspacing=0.15,
          handletextpad=0.35, handlelength=1.2,
          fontsize=14)

save(fig, 'fig3a_f1_flow_vs_window_k10_round30')
plt.close(fig)


## Fig 3b - Flow vs Window30s (Benign FPR, Round 30, k=10)


In [27]:
round_num = 30
k = 10
cats = ['Flow', 'Window 30s']
xs = list(range(len(cats)))
width = 0.36

fig, ax = plt.subplots()

for i, run in enumerate(RUNS):
    flow = read_csv(run, 'eval_fpr_target_flow/f1_scores_fpr_target.csv')
    flow = flow[(flow['granularity']=='flow') & (flow['round']==round_num) & (flow['k']==k)].iloc[0]
    win = read_csv(run, 'eval_fpr_target_window30s/f1_scores_fpr_target.csv')
    win = win[(win['granularity']=='window') & (win['round']==round_num) & (win['k']==k)].iloc[0]
    ys = [float(flow['benign_fpr']*100.0), float(win['benign_fpr']*100.0)]

    offset = (i - 0.5) * width
    ax.bar([x + offset for x in xs], ys, width=width,
           color=run['color'], edgecolor='white', linewidth=0.6,
           label=run['name'])

ax.set_xticks(xs)
ax.set_xticklabels(cats)
ax.set_ylabel('Benign FPR (%)')
ax.grid(True, axis='y', linestyle='--', alpha=0.6)
format_percent_axis(ax, decimals=3)
ax.legend(loc='upper left', bbox_to_anchor=(0.01, 1.02),
          frameon=True, framealpha=0.88,
          borderpad=0.25, labelspacing=0.15,
          handletextpad=0.35, handlelength=1.2,
          fontsize=14)

# Show numeric values (including 0.000 for Window 30s)
for cont in ax.containers:
    ax.bar_label(cont, fmt='%.3f', padding=2, fontsize=12)

save(fig, 'fig3b_benign_fpr_flow_vs_window_k10_round30')
plt.close(fig)


## Fig 4a - Communication per round (MB)


In [16]:
fig, ax = plt.subplots()

for run in RUNS:
    df = read_csv(run, 'communication_metrics.csv').sort_values('round')
    ax.plot(df['round'], df['bytes_total']/1e6,
            color=run['color'], marker='o', label=run['name'])

ax.set_xlabel('Round')
ax.set_ylabel('Communication (MB/round)')
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='upper left', frameon=True)

save(fig, 'fig4a_comm_per_round')
plt.close(fig)


## Fig 4b - Cumulative communication (GB)


In [17]:
fig, ax = plt.subplots()

for run in RUNS:
    df = read_csv(run, 'communication_metrics.csv').sort_values('round')
    cum = df['bytes_total'].cumsum()/1e9
    ax.plot(df['round'], cum,
            color=run['color'], marker='s', label=run['name'])

ax.set_xlabel('Round')
ax.set_ylabel('Cumulative communication (GB)')
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='upper left', frameon=True)

save(fig, 'fig4b_comm_cumulative')
plt.close(fig)
